In [0]:
print('silver customers')

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

###  ***Data Reading***

In [0]:
df = spark.read.format('parquet')\
    .load('abfss://bronze@datalakeete1.dfs.core.windows.net/customers')

In [0]:
df.display()

In [0]:
df = df.drop('_rescued_data')

In [0]:
df.display()

In [0]:
df=df.withColumn('domains',split(col('email'),'@')[1])

In [0]:
df.display()

In [0]:
df.groupBy('domains').agg(count('customer_id').alias('total_customers')).sort('total_customers',ascending=False).display()

In [0]:
import time

In [0]:
df_gmail=df.filter(col('domains')=='gmail.com')
df_gmail.display()
time.sleep(5)

df_yahoo=df.filter(col('domains')=='yahoo.com')
df_yahoo.display()
time.sleep(5)


### Concat

In [0]:
df = df.withColumn('full_name',concat(col('first_name'),lit(' '),col('last_name')))
df = df.drop('first_name','last_name')
df.display()

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .save('abfss://silver@datalakeete1.dfs.core.windows.net/customers')

In [0]:
spark.read.format("delta") \
    .load("abfss://silver@datalakeete1.dfs.core.windows.net/customers") \
    .printSchema()

In [0]:
display(dbutils.fs.ls("abfss://silver@datalakeete1.dfs.core.windows.net/customers"))

In [0]:
from delta.tables import DeltaTable

DeltaTable.forPath(
    spark,
    "abfss://silver@datalakeete1.dfs.core.windows.net/customers"
).history().display()

In [0]:
%python
spark.read.format("delta") \
    .load("abfss://silver@datalakeete1.dfs.core.windows.net/customers") \
    .printSchema()

In [0]:
df.printSchema()

## ### Removing Files

In [0]:
dbutils.fs.rm(
    "abfss://silver@datalakeete1.dfs.core.windows.net/customers",
    True
)

In [0]:
%sql
drop table databricks_cat_1234.silver.customers_silver

In [0]:
%sql
CREATE TABLE databricks_cat_1234.silver.customers_silver
USING DELTA
LOCATION 'abfss://silver@datalakeete1.dfs.core.windows.net/customers'

In [0]:
%sql
select * from databricks_cat_1234.silver.customers_silver limit 4;